In [1]:
import math
import numpy as np
import pandas as pd
from numba import njit, types
from numba.typed import List

# ===================================================================
# 1. Carregamento
# ===================================================================
df_pings_gps_original = pd.read_csv("saida/linha_913_final.csv", parse_dates=["datahora"])
df_paradas_original = pd.read_csv("dados brutos/paradas_913_unificado.csv")

# ===================================================================
# 2. Configurações
# ===================================================================
RAIO_SNAP_M = 100
TIMEOUT_SEC = 30 * 60
MAX_PULOS   = 3

df_pings_gps_original.head(3)

,ordem,latitude,longitude,datahora,linha,temp_max,temp_min,chuva,hora_sin,hora_cos,dia_sin,dia_cos,tipo_dia
0,B10001,-22.81554,-43.18691,2025-12-16 13:08:21,913,38.4,36.2,0.0,-0.258819,-0.965926,0.781831,0.62349,dia_util
1,B10001,-22.81552,-43.18688,2025-12-16 13:38:23,913,38.5,34.7,0.0,-0.258819,-0.965926,0.781831,0.62349,dia_util
2,B10001,-22.81552,-43.18688,2025-12-16 13:43:01,913,38.5,34.7,0.0,-0.258819,-0.965926,0.781831,0.62349,dia_util


In [ ]:
# ===================================================================
# 3. Funções com Numba
# ===================================================================
# Calcula a distância entre dois pontos geográficos
@njit(cache=True)
def haversine_vec(lat, lon, lats, lons):
    R = 6_371_000.0
    n = len(lats)
    dists = np.empty(n)
    for i in range(n):
        dlat = math.radians(lats[i] - lat)
        dlon = math.radians(lons[i] - lon)
        a = (math.sin(dlat / 2) ** 2
             + math.cos(math.radians(lat))
             * math.cos(math.radians(lats[i]))
             * math.sin(dlon / 2) ** 2)
        dists[i] = R * 2 * math.asin(math.sqrt(a))
    return dists

# Snap para a parada mais próxima, considerando sequência da rota
@njit(cache=True)
def snap_idx_contextual(lat, lon, stop_lats, stop_lons, stop_seqs, raio, seq_minima):
    dists = haversine_vec(lat, lon, stop_lats, stop_lons)
    best_idx = -1
    best_dist = 1e18
    for i in range(len(dists)):
        if dists[i] <= raio and stop_seqs[i] > seq_minima:
            if dists[i] < best_dist:
                best_dist = dists[i]
                best_idx = i
    return best_idx

# Snap para a parada mais próxima, sem considerar sequência da rota
@njit(cache=True)
def snap_mais_proximo(lat, lon, stop_lats, stop_lons, raio):
    dists = haversine_vec(lat, lon, stop_lats, stop_lons)
    idx = np.argmin(dists)
    return int(idx) if dists[idx] <= raio else -1

# Processamento de um veículo, retornando os pares de paradas e tempos
@njit(cache=True)
def processar_veiculo(pings_lat, pings_lon, pings_ts, stop_lats, stop_lons, stop_seqs, stop_dirs, raio, timeout_sec, max_pulos):
    idx_a_list = List.empty_list(types.int64)
    idx_b_list = List.empty_list(types.int64)
    t_saida_list = List.empty_list(types.float64)
    t_chegada_list = List.empty_list(types.float64)
    pulos_list = List.empty_list(types.int64)
    
    descartes_mesma_parada, descartes_max_pulos = 0, 0
    estado_idx, estado_ts = -1, 0.0

    for i in range(len(pings_lat)):
        if estado_idx == -1 or pings_ts[i] - estado_ts > timeout_sec:
            s = snap_mais_proximo(pings_lat[i], pings_lon[i], stop_lats, stop_lons, raio)
            if s != -1:
                estado_idx, estado_ts = s, pings_ts[i]
            continue

        seq_minima = stop_seqs[estado_idx]
        s = snap_idx_contextual(pings_lat[i], pings_lon[i], stop_lats, stop_lons, stop_seqs, raio, seq_minima)

        if s == -1:
            if snap_mais_proximo(pings_lat[i], pings_lon[i], stop_lats, stop_lons, raio) == estado_idx:
                descartes_mesma_parada += 1
            continue

        if s == estado_idx:
            descartes_mesma_parada += 1
            continue

        if stop_dirs[s] != stop_dirs[estado_idx] or stop_seqs[s] <= stop_seqs[estado_idx]:
            continue

        pulos = stop_seqs[s] - stop_seqs[estado_idx] - 1
        if pulos > max_pulos:
            descartes_max_pulos += 1
            continue

        idx_a_list.append(estado_idx)
        idx_b_list.append(s)
        t_saida_list.append(estado_ts)
        t_chegada_list.append(pings_ts[i])
        pulos_list.append(pulos)

        estado_idx, estado_ts = s, pings_ts[i]

    return idx_a_list, idx_b_list, t_saida_list, t_chegada_list, pulos_list, descartes_mesma_parada, descartes_max_pulos

# Compilação inicial para ser descartada
_ = processar_veiculo(np.array([0.0]), np.array([0.0]), np.array([0.0]), np.array([0.0]), np.array([0.0]), np.array([0]), np.array([0]), RAIO_SNAP_M, TIMEOUT_SEC, MAX_PULOS)

In [ ]:
# ===================================================================
# 4. Função Principal do Pipeline de Trechos
# ===================================================================
# Recebe os dados brutos e uma lista de paradas a serem ignoradas. Retorna trechos limpos e resumo ordenado da rota.
def pipeline_gerar_trechos(df_gps_raw, df_paradas_raw, lista_remover_paradas):

    # 1. Preparar Paradas
    df_paradas = df_paradas_raw[~df_paradas_raw['stop_id'].isin(lista_remover_paradas)].copy()
    df_paradas = df_paradas.sort_values(['direction_id', 'shape_dist_traveled'])
    df_paradas['stop_sequence'] = df_paradas.groupby('direction_id').cumcount()
    
    stop_lats = df_paradas["stop_lat"].to_numpy(dtype=np.float64)
    stop_lons = df_paradas["stop_lon"].to_numpy(dtype=np.float64)
    stop_seqs = df_paradas["stop_sequence"].to_numpy(dtype=np.int64)
    stop_dirs = df_paradas["direction_id"].to_numpy(dtype=np.int64)

    # 2. Preparar GPS
    df_gps = df_gps_raw.sort_values(["ordem", "datahora"]).copy()
    df_gps["ts"] = df_gps["datahora"].astype(np.int64) / 1e9
    ctx_cols = ["tipo_dia", "hora_cos", "hora_sin", "dia_cos", "dia_sin", "chuva", "temp_max"]

    trechos = []
    
    # 3. Processamento
    for veiculo, grupo in df_gps.groupby("ordem"):
        pings_lat = grupo["latitude"].to_numpy(dtype=np.float64)
        pings_lon = grupo["longitude"].to_numpy(dtype=np.float64)
        pings_ts  = grupo["ts"].to_numpy(dtype=np.float64)
        ctx       = grupo[ctx_cols].to_numpy()

        idx_a, idx_b, t_saida, t_chegada, pulos, _, _ = processar_veiculo(
            pings_lat, pings_lon, pings_ts, stop_lats, stop_lons, stop_seqs, stop_dirs,
            float(RAIO_SNAP_M), float(TIMEOUT_SEC), float(MAX_PULOS)
        )

        for k in range(len(idx_a)):
            ping_chegada_pos = np.searchsorted(pings_ts, t_chegada[k])
            trechos.append({
                "veiculo":      veiculo,
                "direction_id": int(stop_dirs[idx_a[k]]),
                "stop_a":       df_paradas.iloc[idx_a[k]]["stop_id"],
                "stop_b":       df_paradas.iloc[idx_b[k]]["stop_id"],
                "nome_a":       df_paradas.iloc[idx_a[k]]["stop_name"],
                "nome_b":       df_paradas.iloc[idx_b[k]]["stop_name"],
                "t_saida":      pd.Timestamp(t_saida[k],  unit="s"),
                "t_chegada":    pd.Timestamp(t_chegada[k], unit="s"),
                "segundos":     t_chegada[k] - t_saida[k],
                "pulos":        int(pulos[k]),
                **dict(zip(ctx_cols, ctx[min(ping_chegada_pos, len(ctx)-1)]))
            })

    # 4. Construção da planilha final e remoção de outliers
    df_trechos = pd.DataFrame(trechos)
    if df_trechos.empty:
        return pd.DataFrame(), pd.DataFrame()
        
    df_trechos["segmento"] = df_trechos["stop_a"] + "__" + df_trechos["stop_b"]
    df_trechos = df_trechos[df_trechos["segundos"] <= 30 * 60].reset_index(drop=True)

    # Tiramos os outliers de cada segmento, mantendo pelo menos 15 segundos como limite inferior
    limite_inferior = df_trechos.groupby("segmento")["segundos"].transform(lambda x: max(15, x.quantile(0.05)))
    limite_superior = df_trechos.groupby("segmento")["segundos"].transform(lambda x: x.quantile(0.95))
    
    df_trechos = df_trechos[
        (df_trechos["segundos"] >= limite_inferior) & 
        (df_trechos["segundos"] <= limite_superior)
    ].reset_index(drop=True)

    # 5. Resumo ordenado
    resumo_trechos = df_trechos[df_trechos['pulos'] == 0].groupby(
        ['direction_id', 'segmento', 'stop_a', 'stop_b', 'nome_a', 'nome_b']
    ).agg(
        quantidade=('segundos', 'count'),
        tempo_medio_segundos=('segundos', 'mean')
    ).reset_index()

    ordem_paradas = df_paradas[['direction_id', 'stop_id', 'stop_sequence']].drop_duplicates()
    resumo_trechos = resumo_trechos.merge(ordem_paradas, left_on=['direction_id', 'stop_a'], right_on=['direction_id', 'stop_id'], how='left')
    
    resumo_trechos = resumo_trechos.sort_values(by=['direction_id', 'stop_sequence']).reset_index(drop=True)
    resumo_trechos = resumo_trechos.drop(columns=['stop_id', 'stop_sequence'])

    return df_trechos, resumo_trechos

In [4]:
# ===================================================================
# TESTE
# ===================================================================
paradas_para_remover_t1 = [
]

df_trechos_t1, resumo_t1 = pipeline_gerar_trechos(df_pings_gps_original, df_paradas_original, paradas_para_remover_t1)

print(f"Trechos mantidos após outliers: {len(df_trechos_t1)}")
with pd.option_context('display.max_rows', None):
    display(resumo_t1)
    display(df_trechos_t1.head(4))

Trechos mantidos após outliers: 117312


,direction_id,segmento,stop_a,stop_b,nome_a,nome_b,quantidade,tempo_medio_segundos
0,0,3053O00028C2__3053O00002C9,3053O00028C2,3053O00002C9,Ponto Final: Del Castilho :: Linhas para Zona ...,Metrô Del Castilho / Shopping Nova América,3363,598.246506
1,0,3053O00002C9__3050O00014C0,3053O00002C9,3050O00014C0,Metrô Del Castilho / Shopping Nova América,Democráticos - Saída 7,3716,317.465285
2,0,3050O00014C0__3157O00014C0,3050O00014C0,3157O00014C0,Democráticos - Saída 7,Vila do João,5480,237.693613
3,0,3157O00014C0__3157O00019C0,3157O00014C0,3157O00019C0,Vila do João,Clínica da Família Adib Jatene,5347,71.443052
4,0,3157O00019C0__3105O00020C0,3157O00019C0,3105O00020C0,Clínica da Família Adib Jatene,CCMN - Geografia,4533,163.662696
5,0,3105O00020C0__3105O00021C0,3105O00020C0,3105O00021C0,CCMN - Geografia,CENPES - Geografia,3610,43.732410
6,0,3105O00021C0__3105O00010C0,3105O00021C0,3105O00010C0,CENPES - Geografia,CT - CCMN,4048,72.696640
7,0,3105O00010C0__3105O00030C0,3105O00010C0,3105O00030C0,CT - CCMN,CT - Letras,3063,136.792360
8,0,3105O00030C0__3105O00031C0,3105O00030C0,3105O00031C0,CT - Letras,CT - Letras,1551,30.757576
9,0,3105O00031C0__3105O00012C0,3105O00031C0,3105O00012C0,CT - Letras,Belas Artes - Reitoria,3446,87.137551


,veiculo,direction_id,stop_a,stop_b,nome_a,nome_b,t_saida,t_chegada,segundos,pulos,tipo_dia,hora_cos,hora_sin,dia_cos,dia_sin,chuva,temp_max,segmento
0,B10001,0,3053O00028C2,3053O00002C9,Ponto Final: Del Castilho :: Linhas para Zona ...,Metrô Del Castilho / Shopping Nova América,2025-12-16 15:00:10,2025-12-16 15:09:56,586.0,0,dia_util,-0.707107,-0.707107,0.62349,0.781831,0.0,35.1,3053O00028C2__3053O00002C9
1,B10001,0,3053O00002C9,3050O00014C0,Metrô Del Castilho / Shopping Nova América,Democráticos - Saída 7,2025-12-16 15:09:56,2025-12-16 15:14:04,248.0,0,dia_util,-0.707107,-0.707107,0.62349,0.781831,0.0,35.1,3053O00002C9__3050O00014C0
2,B10001,0,3050O00014C0,3157O00014C0,Democráticos - Saída 7,Vila do João,2025-12-16 15:14:04,2025-12-16 15:17:40,216.0,0,dia_util,-0.707107,-0.707107,0.62349,0.781831,0.0,35.1,3050O00014C0__3157O00014C0
3,B10001,0,3157O00014C0,3157O00019C0,Vila do João,Clínica da Família Adib Jatene,2025-12-16 15:17:40,2025-12-16 15:18:41,61.0,0,dia_util,-0.707107,-0.707107,0.62349,0.781831,0.0,35.1,3157O00014C0__3157O00019C0


In [5]:
# ===================================================================
# TESTE 2: Limpeza
# ===================================================================
paradas_para_remover_t2 = [
    '3105O00005C0', 
    '3105O00004C0',
    '3105O00030C0'
]

df_trechos_t2, resumo_t2 = pipeline_gerar_trechos(df_pings_gps_original, df_paradas_original, paradas_para_remover_t2)

print(f"Trechos mantidos após outliers: {len(df_trechos_t2)}")
with pd.option_context('display.max_rows', None):
    display(resumo_t2)

Trechos mantidos após outliers: 116386


,direction_id,segmento,stop_a,stop_b,nome_a,nome_b,quantidade,tempo_medio_segundos
0,0,3053O00028C2__3053O00002C9,3053O00028C2,3053O00002C9,Ponto Final: Del Castilho :: Linhas para Zona ...,Metrô Del Castilho / Shopping Nova América,3406,596.308280
1,0,3053O00002C9__3050O00014C0,3053O00002C9,3050O00014C0,Metrô Del Castilho / Shopping Nova América,Democráticos - Saída 7,3757,316.420282
2,0,3050O00014C0__3157O00014C0,3050O00014C0,3157O00014C0,Democráticos - Saída 7,Vila do João,5534,237.452114
3,0,3157O00014C0__3157O00019C0,3157O00014C0,3157O00019C0,Vila do João,Clínica da Família Adib Jatene,5400,71.415185
4,0,3157O00019C0__3105O00020C0,3157O00019C0,3105O00020C0,Clínica da Família Adib Jatene,CCMN - Geografia,5125,159.720390
5,0,3105O00020C0__3105O00021C0,3105O00020C0,3105O00021C0,CCMN - Geografia,CENPES - Geografia,3763,46.758437
6,0,3105O00021C0__3105O00010C0,3105O00021C0,3105O00010C0,CENPES - Geografia,CT - CCMN,4082,72.605830
7,0,3105O00010C0__3105O00031C0,3105O00010C0,3105O00031C0,CT - CCMN,CT - Letras,5183,145.600039
8,0,3105O00031C0__3105O00012C0,3105O00031C0,3105O00012C0,CT - Letras,Belas Artes - Reitoria,4307,92.693522
9,0,3105O00012C0__3105O00013C0,3105O00012C0,3105O00013C0,Belas Artes - Reitoria,CETEM,1716,42.913170


In [6]:
# Exportar a iteração final
df_trechos_t2.to_csv("saida/trechos_913_final.csv", index=False)